<a href="https://colab.research.google.com/github/noel-odero/Zindi-Health_QA/blob/main/Zindi_health_qa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multilingual Health QA — Retrieval-Augmented Approach

This notebook implements a retrieval-based pipeline for the Zindi Multilingual Health QA competition, covering five African language subsets.


In [3]:
# mount drive and set up required packages
from google.colab import drive
drive.mount('/content/drive/')

!pip install -q sentence-transformers scikit-learn rouge-score tqdm

print('Done')


Mounted at /content/drive/
  Preparing metadata (setup.py) ... done
Done


In [4]:
# imports, constants, and path configuration
import os
import re
import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

DRIVE_DIR = Path('/content/drive/MyDrive/zindi-health-qa')
DATA_DIR  = DRIVE_DIR
OUT_DIR   = DRIVE_DIR / 'outputs'
EMB_DIR   = DRIVE_DIR / 'embeddings'
PRED_DIR  = DRIVE_DIR / 'predictions'
RETRIEVAL_SUB_DIR = DRIVE_DIR / 'retrieval_submissions'

for d in [OUT_DIR, EMB_DIR, PRED_DIR, RETRIEVAL_SUB_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PATH   = DATA_DIR / 'Train.csv'
VAL_PATH     = DATA_DIR / 'Val.csv'
TEST_PATH    = DATA_DIR / 'Test.csv'
SUMMARY_PATH = OUT_DIR / 'experiment_summary_retrieval.csv'

QUESTION_COL = 'input'
ANSWER_COL   = 'output'
LANG_COL     = 'subset'

SEMANTIC_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'  # compact multilingual model with strong cross-lingual transfer
SEMANTIC_BATCH_SIZE = 64

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print('Done')


Device: cuda
Done


In [5]:
# load and clean train/val/test splits
def clean_text(x):
    if pd.isna(x):
        return ''
    return str(x).strip()

train = pd.read_csv(TRAIN_PATH)
val   = pd.read_csv(VAL_PATH)
test  = pd.read_csv(TEST_PATH)

for df in [train, val, test]:
    df[QUESTION_COL] = df[QUESTION_COL].map(clean_text)
    if ANSWER_COL in df.columns:
        df[ANSWER_COL] = df[ANSWER_COL].map(clean_text)

train = train[(train[QUESTION_COL] != '') & (train[ANSWER_COL] != '')].reset_index(drop=True)
val   = val[(val[QUESTION_COL] != '') & (val[ANSWER_COL] != '')].reset_index(drop=True)
test  = test[test[QUESTION_COL] != ''].reset_index(drop=True)

TEST_QUESTION_COL = QUESTION_COL
TEST_LANG_COL     = LANG_COL
TEST_ID_COL       = 'ID'

print(f'Train: {len(train)}')
print(f'Val  : {len(val)}')
print(f'Test : {len(test)}')
print('Done')


Train: 29814
Val  : 6686
Test : 2618
Done


In [6]:
# rouge scoring utilities and resumable experiment tracker
class WhitespaceTokenizer:
    def tokenize(self, text):
        return str(text).strip().split() if text else []

_scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rougeL'], tokenizer=WhitespaceTokenizer(), use_stemmer=False,
)

def compute_rouge(predictions, references):
    r1, rl = [], []
    for pred, ref in zip(predictions, references):
        s = _scorer.score(str(ref), str(pred))
        r1.append(s['rouge1'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return {'rouge1_f1': float(np.mean(r1)), 'rougeL_f1': float(np.mean(rl))}

def evaluate_by_language(predictions, references, languages):
    rows = []
    for lang in sorted(set(languages)):
        mask  = [l == lang for l in languages]
        preds = [p for p, m in zip(predictions, mask) if m]
        refs  = [r for r, m in zip(references, mask) if m]
        m = compute_rouge(preds, refs)
        rows.append({'subset': lang, 'n': len(preds),
                     'ROUGE-1': round(m['rouge1_f1'], 4), 'ROUGE-L': round(m['rougeL_f1'], 4)})
    overall = compute_rouge(predictions, references)
    rows.append({'subset': 'OVERALL', 'n': len(predictions),
                 'ROUGE-1': round(overall['rouge1_f1'], 4), 'ROUGE-L': round(overall['rougeL_f1'], 4)})
    return pd.DataFrame(rows).set_index('subset')

def print_run_summary(label, preds, refs, languages=None, min_answer_len=0):
    m = compute_rouge(preds, refs)
    print(f'{label}: ROUGE-1={m["rouge1_f1"]:.4f}  ROUGE-L={m["rougeL_f1"]:.4f}')
    if languages is not None:
        print(evaluate_by_language(preds, refs, languages).to_string())
    return m

class ExperimentTracker:
    """Tracks experiment results across runs; loads prior rows on init and rewrites on each log() call."""
    def __init__(self, path):
        self.path = path
        if path.exists():
            self.rows = pd.read_csv(path).to_dict('records')
            print(f'Loaded {len(self.rows)} existing experiment records from {path}')
        else:
            self.rows = []

    def is_done(self, experiment_id):
        return any(r['experiment_id'] == experiment_id for r in self.rows)

    def log(self, experiment_id, name, category, change, rationale,
             rouge1, rougel, runtime_min=0.0, notes=''):
        weighted = round(0.37 * rouge1 + 0.37 * rougel, 4)
        self.rows = [r for r in self.rows if r['experiment_id'] != experiment_id]
        entry = {
            'experiment_id': experiment_id, 'name': name, 'category': category,
            'change': change, 'rationale': rationale,
            'rouge1': round(rouge1, 4), 'rougeL': round(rougel, 4), 'weighted': weighted,
            'runtime_min': runtime_min, 'notes': notes,
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M'),
        }
        self.rows.append(entry)
        pd.DataFrame(self.rows).to_csv(self.path, index=False)
        print(f'[{experiment_id}] {name} -> ROUGE-1={rouge1:.4f}  ROUGE-L={rougel:.4f}  weighted={weighted:.4f}')
        return entry

tracker = ExperimentTracker(SUMMARY_PATH)
print('Tracker ready')


Loaded 10 existing experiment records from /content/drive/MyDrive/zindi-health-qa/outputs/experiment_summary_retrieval.csv
Tracker ready


In [7]:
# retrieval index class and inference helpers
def _norm_question_key(q):
    return clean_text(q).lower()

def _minmax_scores(scores):
    scores = np.asarray(scores, dtype=float)
    lo, hi = scores.min(), scores.max()
    if hi - lo < 1e-9:
        return np.ones_like(scores)
    return (scores - lo) / (hi - lo)

DEFAULT_LANGUAGE_STRATEGY = {
    s: ('hybrid', 0.35, 0.65) for s in train[LANG_COL].unique()
}

class SemanticRoutingIndex:
    """Top-1 retrieval index with per-language routing; supports pure TF-IDF, pure semantic, or hybrid scoring."""
    def __init__(
        self, model_name=SEMANTIC_MODEL_NAME, batch_size=SEMANTIC_BATCH_SIZE,
        language_strategy=None, question_col=QUESTION_COL, answer_col=ANSWER_COL,
        id_col='ID', group_col=LANG_COL, tfidf_analyzer='word', tfidf_ngram_range=None,
        tfidf_max_features=200_000, normalize_hybrid_scores=False,
        exact_match_lookup=False, dedup_questions=False, similarity_threshold=None,
        cross_subset_fallback=None, char_tfidf_subsets=None,
        query_encode_prefix='', passage_encode_prefix='',
    ):
        self.model_name = model_name
        self.batch_size = batch_size
        self.language_strategy = language_strategy or DEFAULT_LANGUAGE_STRATEGY
        self.question_col = question_col
        self.answer_col = answer_col
        self.id_col = id_col
        self.group_col = group_col
        self.tfidf_analyzer = tfidf_analyzer
        self.tfidf_ngram_range = tfidf_ngram_range or ((3, 5) if tfidf_analyzer == 'char' else (1, 2))
        self.tfidf_max_features = tfidf_max_features
        self.normalize_hybrid_scores = normalize_hybrid_scores
        self.exact_match_lookup = exact_match_lookup
        self.dedup_questions = dedup_questions
        self.similarity_threshold = similarity_threshold
        self.cross_subset_fallback = cross_subset_fallback or {}
        self.char_tfidf_subsets = set(char_tfidf_subsets or ())
        self.query_encode_prefix = query_encode_prefix
        self.passage_encode_prefix = passage_encode_prefix
        self.encoder = None
        self.questions, self.answers, self.ids = [], [], []
        self.embeddings = None
        self.subset_indices = {}
        self.subset_tfidf = {}
        self.question_to_answer = {}

    def _get_encoder(self):
        if self.encoder is None:
            print(f'  Loading encoder: {self.model_name} ({DEVICE})')
            self.encoder = SentenceTransformer(self.model_name, device=DEVICE)
        return self.encoder

    def _encode_texts(self, texts, prefix=''):
        encoder = self._get_encoder()
        payload = [f'{prefix}{t}' if prefix else t for t in texts]
        parts = []
        for start in range(0, len(payload), self.batch_size):
            end = min(start + self.batch_size, len(payload))
            parts.append(encoder.encode(payload[start:end], show_progress_bar=False, normalize_embeddings=True))
            if end % 5000 == 0 or end == len(payload):
                print(f'    Encoded {end:,} / {len(payload):,}')
        return np.vstack(parts)

    def _make_tfidf_vectorizer(self):
        if self.tfidf_analyzer == 'char':
            return TfidfVectorizer(analyzer='char', ngram_range=self.tfidf_ngram_range, max_features=self.tfidf_max_features)
        return TfidfVectorizer(ngram_range=self.tfidf_ngram_range, max_features=self.tfidf_max_features)

    def fit(self, df, cached_embeddings=None):
        work = df.copy()
        if self.dedup_questions:
            work['_q_norm'] = work[self.question_col].map(clean_text)
            work = work.drop_duplicates(subset=[self.group_col, '_q_norm'], keep='first').drop(columns=['_q_norm']).reset_index(drop=True)

        self.questions = work[self.question_col].fillna('').astype(str).tolist()
        self.answers = work[self.answer_col].fillna('').astype(str).tolist()
        self.ids = work[self.id_col].fillna('').astype(str).tolist() if self.id_col in work.columns else [''] * len(self.questions)

        if cached_embeddings is not None and len(cached_embeddings) == len(self.questions):
            self.embeddings = cached_embeddings
            print(f'  Using cached embeddings for {len(self.questions):,} questions')
        else:
            print(f'  Encoding {len(self.questions):,} questions...')
            self.embeddings = self._encode_texts(self.questions, prefix=self.passage_encode_prefix)

        if self.exact_match_lookup:
            self.question_to_answer = {}
            for q, a in zip(self.questions, self.answers):
                key = _norm_question_key(q)
                if key and key not in self.question_to_answer:
                    self.question_to_answer[key] = a

        self.subset_indices = {}
        self.subset_tfidf = {}
        for subset_code in work[self.group_col].unique():
            mask = work[self.group_col] == subset_code
            indices = np.where(mask.values)[0]
            self.subset_indices[subset_code] = indices
            subset_questions = [self.questions[i] for i in indices]
            vectorizer = (
                TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=self.tfidf_max_features)
                if subset_code in self.char_tfidf_subsets else self._make_tfidf_vectorizer()
            )
            matrix = vectorizer.fit_transform(subset_questions)
            self.subset_tfidf[subset_code] = {'vectorizer': vectorizer, 'matrix': matrix}
        print(f'  Built index: {len(self.questions):,} rows, {len(self.subset_indices)} subsets')
        return self

    def _search_subsets(self, subset):
        if subset not in self.subset_indices:
            subset = next(iter(self.subset_indices))
        subsets = [subset]
        fallback = self.cross_subset_fallback.get(subset)
        if fallback and fallback in self.subset_indices and fallback not in subsets:
            subsets.append(fallback)
        return subsets

    def _collect_candidates(self, subsets, exclude_id=None):
        keep_global, keep_local_by_subset, seen = [], {}, set()
        for sub in subsets:
            all_indices = self.subset_indices[sub]
            keep_local = []
            for local_row, global_idx in enumerate(all_indices):
                if exclude_id is not None and self.ids[global_idx] == str(exclude_id):
                    continue
                gi = int(global_idx)
                if gi in seen:
                    continue
                seen.add(gi)
                keep_local.append(local_row)
                keep_global.append(gi)
            keep_local_by_subset[sub] = keep_local
        if not keep_global:
            sub = subsets[0]
            keep_global = [int(i) for i in self.subset_indices[sub]]
            keep_local_by_subset[sub] = list(range(len(keep_global)))
        return np.array(keep_global, dtype=int), keep_local_by_subset, subsets

    def retrieve_one(self, question, question_embedding, subset, exclude_id=None):
        if self.exact_match_lookup:
            hit = self.question_to_answer.get(_norm_question_key(question))
            if hit is not None:
                return hit
        subsets = self._search_subsets(subset)
        primary_subset = subsets[0]
        method, tfidf_w, semantic_w = self.language_strategy.get(primary_subset, ('hybrid', 0.35, 0.65))
        candidate_indices, keep_local_by_subset, subsets = self._collect_candidates(subsets, exclude_id=exclude_id)
        candidate_embeddings = self.embeddings[candidate_indices]
        semantic_scores = cosine_similarity(question_embedding.reshape(1, -1), candidate_embeddings).flatten()

        if method == 'semantic' or tfidf_w == 0.0:
            best_idx = int(candidate_indices[int(np.argmax(semantic_scores))])
            return self.answers[best_idx]

        subset_info = self.subset_tfidf[primary_subset]
        query_tfidf = subset_info['vectorizer'].transform([question])
        tfidf_scores = np.zeros(len(candidate_indices), dtype=float)
        offset = 0
        for sub in subsets:
            local_rows = keep_local_by_subset.get(sub, [])
            if not local_rows:
                continue
            n = len(local_rows)
            if sub == primary_subset:
                block = cosine_similarity(query_tfidf, subset_info['matrix'][local_rows]).flatten()
            else:
                fb_info = self.subset_tfidf[sub]
                block = cosine_similarity(fb_info['vectorizer'].transform([question]), fb_info['matrix'][local_rows]).flatten()
            tfidf_scores[offset:offset + n] = block
            offset += n

        if self.normalize_hybrid_scores:
            semantic_scores = _minmax_scores(semantic_scores)
            tfidf_scores = _minmax_scores(tfidf_scores)
        if self.similarity_threshold is not None:
            if float(np.max(semantic_scores)) < self.similarity_threshold:
                tfidf_w, semantic_w = 1.0, 0.0

        hybrid_scores = tfidf_w * tfidf_scores + semantic_w * semantic_scores
        best_local = int(np.argmax(hybrid_scores))
        return self.answers[int(candidate_indices[best_local])]

    def predict_dataframe(self, df, question_col, group_col, id_col=None,
                            question_embeddings=None, desc='Routing', log_every=0, references=None):
        questions = df[question_col].fillna('').astype(str).tolist()
        subsets = df[group_col].tolist()
        ids = df[id_col].fillna('').astype(str).tolist() if id_col and id_col in df.columns else [None] * len(df)
        if question_embeddings is None:
            print(f'  Encoding {len(questions):,} query embeddings...')
            question_embeddings = self._encode_texts(questions, prefix=self.query_encode_prefix)
        predictions = []
        pairs = list(zip(questions, subsets, ids, question_embeddings))
        row_iter = tqdm(pairs, total=len(pairs), desc=desc) if len(pairs) > 100 else pairs
        for i, (question, subset, row_id, emb) in enumerate(row_iter):
            predictions.append(self.retrieve_one(question, emb, subset, exclude_id=row_id))
            if references is not None and log_every and (i + 1) % log_every == 0:
                partial = compute_rouge(predictions, references[:len(predictions)])
                msg = f'  [{i+1:,}/{len(df):,}] R1={partial["rouge1_f1"]:.4f} RL={partial["rougeL_f1"]:.4f}'
                (tqdm.write(msg) if len(pairs) > 100 else print(msg))
        return predictions

def encode_questions_for_model(model_name, texts, batch_size=SEMANTIC_BATCH_SIZE, query_prefix=''):
    encoder = SentenceTransformer(model_name, device=DEVICE)
    payload = [f'{query_prefix}{t}' if query_prefix else t for t in texts]
    parts = []
    for start in range(0, len(payload), batch_size):
        end = min(start + batch_size, len(payload))
        parts.append(encoder.encode(payload[start:end], show_progress_bar=False, normalize_embeddings=True))
    return np.vstack(parts)

def make_submission_fn(ids, preds, output_path):
    clean_preds = [str(p).strip() if str(p).strip() else 'Please consult a qualified health professional.' for p in preds]
    sub = pd.DataFrame({
        'ID': ids, 'TargetRLF1': clean_preds, 'TargetR1F1': clean_preds, 'TargetLLM': clean_preds,
    })
    assert len(sub) == len(ids)
    assert sub[['TargetRLF1','TargetR1F1','TargetLLM']].notna().all().all()
    sub.to_csv(output_path, index=False, encoding='utf-8')
    print(f'  Submission saved: {output_path}  ({len(sub)} rows)')

def save_test_submission(index, eid, use_lang_tag_note=''):
    """Build and save a Zindi-format test submission for a given experiment.
    Each experiment gets its own file so leaderboard progression can be tracked per run."""
    test_qs = test[TEST_QUESTION_COL].fillna('').astype(str).tolist()
    test_emb_path = EMB_DIR / 'test_question_embeddings.npy'
    if test_emb_path.exists():
        test_embeddings = np.load(test_emb_path)
    else:
        test_embeddings = encode_questions_for_model(SEMANTIC_MODEL_NAME, test_qs)
        np.save(test_emb_path, test_embeddings)

    test_preds = index.predict_dataframe(
        test, question_col=TEST_QUESTION_COL, group_col=TEST_LANG_COL, id_col=None,
        question_embeddings=test_embeddings, desc=f'{eid} test',
    )
    sub_path = RETRIEVAL_SUB_DIR / f'{eid}_submission.csv'
    make_submission_fn(test[TEST_ID_COL].values, test_preds, sub_path)
    return sub_path

print('SemanticRoutingIndex and helpers ready')


SemanticRoutingIndex and helpers ready


In [13]:
# encode and cache embeddings once to avoid redundant computation
TRAIN_EMB_PATH = EMB_DIR / 'train_question_embeddings.npy'
VAL_EMB_PATH   = EMB_DIR / 'val_question_embeddings.npy'
TEST_EMB_PATH  = EMB_DIR / 'test_question_embeddings.npy'

if TRAIN_EMB_PATH.exists():
    train_embeddings = np.load(TRAIN_EMB_PATH)
    print(f'Loaded cached train embeddings: {train_embeddings.shape}')
else:
    print('Encoding train question embeddings (one-time cost)...')
    t0 = time.time()
    train_embeddings = encode_questions_for_model(SEMANTIC_MODEL_NAME, train[QUESTION_COL].tolist())
    np.save(TRAIN_EMB_PATH, train_embeddings)
    print(f'Done in {(time.time()-t0)/60:.1f} min, saved to {TRAIN_EMB_PATH}')

if VAL_EMB_PATH.exists():
    val_embeddings = np.load(VAL_EMB_PATH)
    print(f'Loaded cached val embeddings: {val_embeddings.shape}')
else:
    print('Encoding val question embeddings...')
    t0 = time.time()
    val_embeddings = encode_questions_for_model(SEMANTIC_MODEL_NAME, val[QUESTION_COL].tolist())
    np.save(VAL_EMB_PATH, val_embeddings)
    print(f'Done in {(time.time()-t0)/60:.1f} min, saved to {VAL_EMB_PATH}')

print('Embeddings ready - reused by every experiment below')


Loaded cached train embeddings: (29814, 384)
Loaded cached val embeddings: (6686, 384)
Embeddings ready - reused by every experiment below


In [14]:
import shutil

old_summary = SUMMARY_PATH
if old_summary.exists():
    backup = OUT_DIR / 'experiment_summary_OLD_mt5_approach.csv'
    shutil.move(str(old_summary), str(backup))
    print(f'Moved old summary to {backup}')

# Reinitialize tracker fresh
tracker = ExperimentTracker(SUMMARY_PATH)

In [15]:
# experiment 1: TF-IDF word n-grams only
EID = 'E01'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    strategy = {s: ('tfidf', 1.0, 0.0) for s in train[LANG_COL].unique()}
    index = SemanticRoutingIndex(language_strategy=strategy, tfidf_analyzer='word').fit(
        train, cached_embeddings=train_embeddings
    )
    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'TF-IDF word n-grams only', 'lexical',
        change='Pure TF-IDF (word 1-2 grams) retrieval per subset, no semantic component',
        rationale='Provides a fast lexical-only reference point before introducing semantic components',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Using cached embeddings for 29,814 questions
  Built index: 29,814 rows, 8 subsets


E01:   0%|          | 0/6686 [00:00<?, ?it/s]

E01 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E01_submission.csv  (2618 rows)
E01: ROUGE-1=0.3927  ROUGE-L=0.3360
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2796   0.1673
Amh_Eth   462   0.1454   0.1365
Eng_Eth   564   0.5373   0.5202
Eng_Gha  1104   0.2557   0.1723
Eng_Ken   390   0.5472   0.5018
Eng_Uga  1688   0.4817   0.4294
Lug_Uga   846   0.4246   0.3995
Swa_Ken   518   0.5321   0.4914
OVERALL  6686   0.3927   0.3360
[E01] TF-IDF word n-grams only -> ROUGE-1=0.3927  ROUGE-L=0.3360  weighted=0.2696


In [23]:
# experiment 2: TF-IDF character n-grams only
EID = 'E02'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    strategy = {s: ('tfidf', 1.0, 0.0) for s in train[LANG_COL].unique()}
    index = SemanticRoutingIndex(language_strategy=strategy, tfidf_analyzer='char').fit(
        train, cached_embeddings=train_embeddings
    )
    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'TF-IDF char n-grams only', 'lexical',
        change='TF-IDF char(3,5) n-grams instead of word n-grams',
        rationale='Sub-word character n-grams are better suited to agglutinative African languages where whole-word TF-IDF misses morphological variants',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Using cached embeddings for 29,814 questions
  Built index: 29,814 rows, 8 subsets


E02:   0%|          | 0/6686 [00:00<?, ?it/s]

E02 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E02_submission.csv  (2618 rows)
E02: ROUGE-1=0.4191  ROUGE-L=0.3630
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2856   0.1697
Amh_Eth   462   0.1509   0.1413
Eng_Eth   564   0.5360   0.5185
Eng_Gha  1104   0.2615   0.1739
Eng_Ken   390   0.5932   0.5547
Eng_Uga  1688   0.5103   0.4616
Lug_Uga   846   0.5037   0.4805
Swa_Ken   518   0.5878   0.5532
OVERALL  6686   0.4191   0.3630
[E02] TF-IDF char n-grams only -> ROUGE-1=0.4191  ROUGE-L=0.3630  weighted=0.2894


In [24]:
# experiment 3: Semantic only - sentence embeddings
EID = 'E03'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    strategy = {s: ('semantic', 0.0, 1.0) for s in train[LANG_COL].unique()}
    index = SemanticRoutingIndex(language_strategy=strategy).fit(
        train, cached_embeddings=train_embeddings
    )
    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'Semantic embeddings only', 'semantic',
        change='Pure multilingual sentence-embedding cosine similarity retrieval, no TF-IDF',
        rationale='Semantic embeddings capture meaning beyond surface form; useful when questions are paraphrased or phrased differently from training data',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Using cached embeddings for 29,814 questions
  Built index: 29,814 rows, 8 subsets


E03:   0%|          | 0/6686 [00:00<?, ?it/s]

E03 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E03_submission.csv  (2618 rows)
E03: ROUGE-1=0.4134  ROUGE-L=0.3608
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2581   0.1561
Amh_Eth   462   0.1164   0.1079
Eng_Eth   564   0.5498   0.5300
Eng_Gha  1104   0.2694   0.1767
Eng_Ken   390   0.7453   0.7242
Eng_Uga  1688   0.6584   0.6220
Lug_Uga   846   0.2392   0.2092
Swa_Ken   518   0.4069   0.3581
OVERALL  6686   0.4134   0.3608
[E03] Semantic embeddings only -> ROUGE-1=0.4134  ROUGE-L=0.3608  weighted=0.2865


In [16]:
# experiment 4: Hybrid 50/50 (TF-IDF + semantic)
EID = 'E04'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    strategy = {s: ('hybrid', 0.5, 0.5) for s in train[LANG_COL].unique()}
    index = SemanticRoutingIndex(language_strategy=strategy, tfidf_analyzer='char').fit(
        train, cached_embeddings=train_embeddings
    )
    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'Hybrid 50/50 TF-IDF + semantic', 'hybrid',
        change='Equal-weight blend of char TF-IDF and semantic cosine similarity scores',
        rationale='Neither lexical nor semantic alone may be optimal; blending both signals should improve retrieval robustness across languages',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Using cached embeddings for 29,814 questions
  Built index: 29,814 rows, 8 subsets


E04:   0%|          | 0/6686 [00:00<?, ?it/s]

E04 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E04_submission.csv  (2618 rows)
E04: ROUGE-1=0.4453  ROUGE-L=0.3889
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2869   0.1713
Amh_Eth   462   0.1600   0.1489
Eng_Eth   564   0.5495   0.5304
Eng_Gha  1104   0.2802   0.1872
Eng_Ken   390   0.6998   0.6704
Eng_Uga  1688   0.5816   0.5369
Lug_Uga   846   0.4754   0.4492
Swa_Ken   518   0.5937   0.5545
OVERALL  6686   0.4453   0.3889
[E04] Hybrid 50/50 TF-IDF + semantic -> ROUGE-1=0.4453  ROUGE-L=0.3889  weighted=0.3087


In [17]:
# experiment 5: Per-language hybrid weight tuning (grid search)
EID = 'E05'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    base_strategy = {s: ('hybrid', 0.5, 0.5) for s in train[LANG_COL].unique()}
    index = SemanticRoutingIndex(language_strategy=base_strategy, tfidf_analyzer='char').fit(
        train, cached_embeddings=train_embeddings
    )

    tuned_strategy = dict(index.language_strategy)
    weight_candidates = (0.0, 0.25, 0.5, 0.75, 1.0)
    for subset in train[LANG_COL].unique():
        mask = (val[LANG_COL] == subset).values
        if not mask.any():
            continue
        subset_val = val.loc[mask].reset_index(drop=True)
        subset_emb = val_embeddings[mask]
        best_w, best_r1 = tuned_strategy[subset][1], -1.0
        for tfidf_w in weight_candidates:
            semantic_w = 1.0 - tfidf_w
            method = 'semantic' if tfidf_w == 0.0 else ('tfidf' if semantic_w == 0.0 else 'hybrid')
            trial = dict(index.language_strategy)
            trial[subset] = (method, float(tfidf_w), float(semantic_w))
            index.language_strategy = trial
            p = index.predict_dataframe(
                subset_val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
                question_embeddings=subset_emb, desc=f'tune {subset}',
            )
            r1 = compute_rouge(p, subset_val[ANSWER_COL].tolist())['rouge1_f1']
            if r1 > best_r1:
                best_r1, best_w = r1, tfidf_w
        semantic_w = 1.0 - best_w
        method = 'semantic' if best_w == 0.0 else ('tfidf' if semantic_w == 0.0 else 'hybrid')
        tuned_strategy[subset] = (method, float(best_w), float(semantic_w))
        print(f'  {subset}: best tfidf_w={best_w:.2f} (R1={best_r1:.4f})')

    index.language_strategy = tuned_strategy
    with open(EMB_DIR / 'tuned_strategy.json', 'w') as f:
        json.dump({k: list(v) for k, v in tuned_strategy.items()}, f, indent=2)

    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'Per-language hybrid weight tuning', 'hybrid',
        change='Grid-searched tfidf_w per subset over {0, .25, .5, .75, 1} instead of fixed 50/50',
        rationale='A fixed global blend ignores language-specific retrieval dynamics; tuning per subset allows the model to adapt to each language\'s structure',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
        notes=f'Tuned weights saved to {EMB_DIR}/tuned_strategy.json',
    )


  Using cached embeddings for 29,814 questions
  Built index: 29,814 rows, 8 subsets


tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

tune Aka_Gha:   0%|          | 0/1114 [00:00<?, ?it/s]

...

tune Swa_Ken:   0%|          | 0/518 [00:00<?, ?it/s]

  Swa_Ken: best tfidf_w=0.75 (R1=0.6186)


E05:   0%|          | 0/6686 [00:00<?, ?it/s]

E05 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E05_submission.csv  (2618 rows)
E05: ROUGE-1=0.4734  ROUGE-L=0.4200
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2881   0.1717
Amh_Eth   462   0.1600   0.1489
Eng_Eth   564   0.5498   0.5300
Eng_Gha  1104   0.2802   0.1872
Eng_Ken   390   0.7453   0.7242
Eng_Uga  1688   0.6584   0.6220
Lug_Uga   846   0.5067   0.4820
Swa_Ken   518   0.6186   0.5833
OVERALL  6686   0.4734   0.4200
[E05] Per-language hybrid weight tuning -> ROUGE-1=0.4734  ROUGE-L=0.4200  weighted=0.3306


In [18]:
# experiment 6: Exact-match lookup layer on top of tuned hybrid
EID = 'E06'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    with open(EMB_DIR / 'tuned_strategy.json') as f:
        tuned_raw = json.load(f)
    tuned_strategy = {k: tuple(v) for k, v in tuned_raw.items()}

    index = SemanticRoutingIndex(
        language_strategy=tuned_strategy, tfidf_analyzer='char', exact_match_lookup=True,
    ).fit(train, cached_embeddings=train_embeddings)

    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'Exact-match lookup + tuned hybrid', 'hybrid',
        change='Added an exact-match question lookup as a fast-path before falling back to hybrid retrieval (E5 weights)',
        rationale='Exact-match questions can be answered with certainty; bypassing the similarity ranking avoids introducing retrieval noise for known queries',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Using cached embeddings for 29,814 questions
  Built index: 29,814 rows, 8 subsets


E06:   0%|          | 0/6686 [00:00<?, ?it/s]

E06 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E06_submission.csv  (2618 rows)
E06: ROUGE-1=0.4730  ROUGE-L=0.4195
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2881   0.1717
Amh_Eth   462   0.1600   0.1489
Eng_Eth   564   0.5474   0.5274
Eng_Gha  1104   0.2802   0.1872
Eng_Ken   390   0.7453   0.7242
Eng_Uga  1688   0.6574   0.6210
Lug_Uga   846   0.5067   0.4820
Swa_Ken   518   0.6186   0.5833
OVERALL  6686   0.4730   0.4195
[E06] Exact-match lookup + tuned hybrid -> ROUGE-1=0.4730  ROUGE-L=0.4195  weighted=0.3302


In [19]:
# experiment 7: Deduplicated corpus
EID = 'E07'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    with open(EMB_DIR / 'tuned_strategy.json') as f:
        tuned_raw = json.load(f)
    tuned_strategy = {k: tuple(v) for k, v in tuned_raw.items()}

    index = SemanticRoutingIndex(
        language_strategy=tuned_strategy, tfidf_analyzer='char',
        exact_match_lookup=True, dedup_questions=True,
    ).fit(train, cached_embeddings=train_embeddings)

    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'Deduplicated corpus', 'data',
        change='Drop duplicate (subset, question) pairs from the index before fitting',
        rationale='Duplicate questions with inconsistent answers add noise to the retrieval corpus; removing them improves answer quality for matched queries',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Encoding 28,834 questions...
  Loading encoder: paraphrase-multilingual-MiniLM-L12-v2 (cuda)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    Encoded 28,834 / 28,834
  Built index: 28,834 rows, 8 subsets


E07:   0%|          | 0/6686 [00:00<?, ?it/s]

E07 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E07_submission.csv  (2618 rows)
E07: ROUGE-1=0.4730  ROUGE-L=0.4195
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2881   0.1717
Amh_Eth   462   0.1600   0.1489
Eng_Eth   564   0.5477   0.5276
Eng_Gha  1104   0.2802   0.1872
Eng_Ken   390   0.7453   0.7242
Eng_Uga  1688   0.6574   0.6210
Lug_Uga   846   0.5067   0.4820
Swa_Ken   518   0.6186   0.5833
OVERALL  6686   0.4730   0.4195
[E07] Deduplicated corpus -> ROUGE-1=0.4730  ROUGE-L=0.4195  weighted=0.3302


In [29]:
# experiment 8: Cross-subset fallback for low-resource languages
EID = 'E08'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    with open(EMB_DIR / 'tuned_strategy.json') as f:
        tuned_raw = json.load(f)
    tuned_strategy = {k: tuple(v) for k, v in tuned_raw.items()}

    # Amharic and Luganda have the fewest training rows - let them fall back to
    # the next-closest English subset (Ethiopia / Uganda) when no good local match exists
    fallback_map = {
        'Amh_Eth': 'Eng_Eth',
        'Lug_Uga': 'Eng_Uga',
    }

    index = SemanticRoutingIndex(
        language_strategy=tuned_strategy, tfidf_analyzer='char',
        exact_match_lookup=True, dedup_questions=True,
        cross_subset_fallback=fallback_map,
    ).fit(train, cached_embeddings=train_embeddings)

    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'Cross-subset fallback for low-resource languages', 'data',
        change='Amh_Eth and Lug_Uga additionally search their corresponding English subset as backup candidates',
        rationale='Low-resource subsets have a smaller candidate pool; allowing fallback to a related English subset can surface relevant answers when in-language coverage is thin',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Encoding 28,834 questions...
  Loading encoder: paraphrase-multilingual-MiniLM-L12-v2 (cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    Encoded 28,834 / 28,834
  Built index: 28,834 rows, 8 subsets


E08:   0%|          | 0/6686 [00:00<?, ?it/s]

E08 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E08_submission.csv  (2618 rows)
E08: ROUGE-1=0.4728  ROUGE-L=0.4194
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2881   0.1717
Amh_Eth   462   0.1576   0.1468
Eng_Eth   564   0.5477   0.5276
Eng_Gha  1104   0.2802   0.1872
Eng_Ken   390   0.7453   0.7242
Eng_Uga  1688   0.6574   0.6210
Lug_Uga   846   0.5068   0.4821
Swa_Ken   518   0.6186   0.5833
OVERALL  6686   0.4728   0.4194
[E08] Cross-subset fallback for low-resource languages -> ROUGE-1=0.4728  ROUGE-L=0.4194  weighted=0.3301


In [20]:
# experiment 9: Similarity threshold fallback (force TF-IDF when semantic is weak)
EID = 'E09'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    with open(EMB_DIR / 'tuned_strategy.json') as f:
        tuned_raw = json.load(f)
    tuned_strategy = {k: tuple(v) for k, v in tuned_raw.items()}

    index = SemanticRoutingIndex(
        language_strategy=tuned_strategy, tfidf_analyzer='char',
        exact_match_lookup=True, dedup_questions=True,
        similarity_threshold=0.3,
    ).fit(train, cached_embeddings=train_embeddings)

    preds = index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        question_embeddings=val_embeddings, desc=EID,
    )
    pd.DataFrame({'ID': val['ID'], 'prediction': preds}).to_csv(PRED_DIR / f'{EID}_val_preds.csv', index=False)
    save_test_submission(index, EID)
    metrics = print_run_summary(EID, preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())
    tracker.log(
        EID, 'Similarity threshold fallback', 'hybrid',
        change='If best semantic similarity < 0.3, force pure TF-IDF for that query instead of trusting a weak semantic match',
        rationale='Low semantic similarity scores indicate the query may be out-of-distribution; switching to TF-IDF in those cases avoids over-trusting a weak embedding match',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
    )


  Encoding 28,834 questions...
  Loading encoder: paraphrase-multilingual-MiniLM-L12-v2 (cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    Encoded 28,834 / 28,834
  Built index: 28,834 rows, 8 subsets


E09:   0%|          | 0/6686 [00:00<?, ?it/s]

E09 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/retrieval_submissions/E09_submission.csv  (2618 rows)
E09: ROUGE-1=0.4730  ROUGE-L=0.4195
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   0.2881   0.1717
Amh_Eth   462   0.1600   0.1489
Eng_Eth   564   0.5477   0.5276
Eng_Gha  1104   0.2802   0.1872
Eng_Ken   390   0.7453   0.7242
Eng_Uga  1688   0.6574   0.6210
Lug_Uga   846   0.5067   0.4820
Swa_Ken   518   0.6186   0.5833
OVERALL  6686   0.4730   0.4195
[E09] Similarity threshold fallback -> ROUGE-1=0.4730  ROUGE-L=0.4195  weighted=0.3302


In [21]:
print(list(OUT_DIR.iterdir()))

[PosixPath('/content/drive/MyDrive/zindi-health-qa/outputs/E10_final_submission.csv'), PosixPath('/content/drive/MyDrive/zindi-health-qa/outputs/experiment_summary_OLD_mt5_approach.csv'), PosixPath('/content/drive/MyDrive/zindi-health-qa/outputs/experiment_summary_retrieval.csv')]


In [22]:
print(tracker.rows)
print(len(tracker.rows))

[{'experiment_id': 'E01', 'name': 'TF-IDF word n-grams only', 'category': 'lexical', 'change': 'Pure TF-IDF (word 1-2 grams) retrieval per subset, no semantic component', 'rationale': 'Provides a fast lexical-only reference point before introducing semantic components', 'rouge1': 0.3927, 'rougeL': 0.336, 'weighted': 0.2696, 'runtime_min': 3.9, 'notes': '', 'timestamp': '2026-06-25 17:16'}, {'experiment_id': 'E04', 'name': 'Hybrid 50/50 TF-IDF + semantic', 'category': 'hybrid', 'change': 'Equal-weight blend of char TF-IDF and semantic cosine similarity scores', 'rationale': 'Neither lexical nor semantic alone may be optimal; blending both signals should improve retrieval robustness across languages', 'rouge1': 0.4453, 'rougeL': 0.3889, 'weighted': 0.3087, 'runtime_min': 10.1, 'notes': '', 'timestamp': '2026-06-25 17:27'}, {'experiment_id': 'E05', 'name': 'Per-language hybrid weight tuning', 'category': 'hybrid', 'change': 'Grid-searched tfidf_w per subset over {0, .25, .5, .75, 1} inste

In [23]:
pd.DataFrame(tracker.rows).to_csv(SUMMARY_PATH, index=False)
print(f'Saved to {SUMMARY_PATH}')
print(list(OUT_DIR.iterdir()))

Saved to /content/drive/MyDrive/zindi-health-qa/outputs/experiment_summary_retrieval.csv
[PosixPath('/content/drive/MyDrive/zindi-health-qa/outputs/E10_final_submission.csv'), PosixPath('/content/drive/MyDrive/zindi-health-qa/outputs/experiment_summary_OLD_mt5_approach.csv'), PosixPath('/content/drive/MyDrive/zindi-health-qa/outputs/experiment_summary_retrieval.csv')]


In [24]:
print(tracker.rows)

[{'experiment_id': 'E01', 'name': 'TF-IDF word n-grams only', 'category': 'lexical', 'change': 'Pure TF-IDF (word 1-2 grams) retrieval per subset, no semantic component', 'rationale': 'Provides a fast lexical-only reference point before introducing semantic components', 'rouge1': 0.3927, 'rougeL': 0.336, 'weighted': 0.2696, 'runtime_min': 3.9, 'notes': '', 'timestamp': '2026-06-25 17:16'}, {'experiment_id': 'E04', 'name': 'Hybrid 50/50 TF-IDF + semantic', 'category': 'hybrid', 'change': 'Equal-weight blend of char TF-IDF and semantic cosine similarity scores', 'rationale': 'Neither lexical nor semantic alone may be optimal; blending both signals should improve retrieval robustness across languages', 'rouge1': 0.4453, 'rougeL': 0.3889, 'weighted': 0.3087, 'runtime_min': 10.1, 'notes': '', 'timestamp': '2026-06-25 17:27'}, {'experiment_id': 'E05', 'name': 'Per-language hybrid weight tuning', 'category': 'hybrid', 'change': 'Grid-searched tfidf_w per subset over {0, .25, .5, .75, 1} inste

In [25]:
tracker.rows = [r for r in tracker.rows if r['experiment_id'] != 'E10']
pd.DataFrame(tracker.rows).to_csv(SUMMARY_PATH, index=False)
print('E10 removed')
print(tracker.is_done('E10'))

E10 removed
False


In [26]:
# experiment 10: Final model - best config combined, indexed on train+val, test submission
EID = 'E10'
if tracker.is_done(EID):
    print(f'{EID} already done, skipping')
else:
    t0 = time.time()
    summary = pd.read_csv(SUMMARY_PATH)
    best_row = summary.loc[summary['rouge1'].idxmax()]
    print('Best experiment so far:')
    print(best_row.to_string())

    with open(EMB_DIR / 'tuned_strategy.json') as f:
        tuned_raw = json.load(f)
    tuned_strategy = {k: tuple(v) for k, v in tuned_raw.items()}

    fallback_map = {'Amh_Eth': 'Eng_Eth', 'Lug_Uga': 'Eng_Uga'}

    # Combine winning settings: tuned weights, exact match, dedup, cross-subset
    # fallback, similarity threshold - all of which were >= baseline in testing
    corpus = pd.concat([train, val], ignore_index=True)

    final_index = SemanticRoutingIndex(
        language_strategy=tuned_strategy, tfidf_analyzer='char',
        exact_match_lookup=True, dedup_questions=True,
        cross_subset_fallback=fallback_map, similarity_threshold=0.3,
    ).fit(corpus)  # full corpus needs fresh embeddings (train+val combined)

    # Sanity check on val first (val is now IN the corpus, so exclude_id handles leakage)
    val_check_preds = final_index.predict_dataframe(
        val, question_col=QUESTION_COL, group_col=LANG_COL, id_col='ID',
        desc='E10 val-check',
    )
    metrics = print_run_summary('E10 (val sanity check)', val_check_preds, val[ANSWER_COL].tolist(), languages=val[LANG_COL].tolist())

    if metrics['rouge1_f1'] > 0.85:
        print('\nWARNING: ROUGE unexpectedly high - possible self-match leakage.')
        print('Verify exact_match_lookup respects exclude_id before trusting this run.\n')

    print('\nGenerating test predictions and submission...')
    test_qs = test[TEST_QUESTION_COL].fillna('').astype(str).tolist()
    test_embeddings = encode_questions_for_model(SEMANTIC_MODEL_NAME, test_qs)
    np.save(TEST_EMB_PATH, test_embeddings)

    test_preds = final_index.predict_dataframe(
        test, question_col=TEST_QUESTION_COL, group_col=TEST_LANG_COL, id_col=None,
        question_embeddings=test_embeddings, desc='E10 test',
    )
    make_submission_fn(test[TEST_ID_COL].values, test_preds, OUT_DIR / f'{EID}_final_submission.csv')

    tracker.log(
        EID, 'Final combined model (train+val corpus)', 'final',
        change='Combined all winning settings from E1-E9 (tuned per-language weights, exact match, dedup, cross-subset fallback, similarity threshold) and re-indexed on train+val combined',
        rationale='Combining train and val into a single retrieval corpus maximizes coverage, increasing the chance of finding a high-quality match for each test query',
        rouge1=metrics['rouge1_f1'], rougel=metrics['rougeL_f1'],
        runtime_min=round((time.time()-t0)/60, 1),
        notes='val sanity-check score shown (val included in corpus with self-exclusion); see E10_final_submission.csv for actual Zindi submission',
    )


Best experiment so far:
experiment_id                                                  E05
name                             Per-language hybrid weight tuning
category                                                    hybrid
change           Grid-searched tfidf_w per subset over {0, .25,...
rationale        A fixed global blend ignores language-specific...
rouge1                                                      0.4734
rougeL                                                        0.42
weighted                                                    0.3306
runtime_min                                                   31.8
notes            Tuned weights saved to /content/drive/MyDrive/...
timestamp                                         2026-06-25 17:59
  Encoding 35,454 questions...
  Loading encoder: paraphrase-multilingual-MiniLM-L12-v2 (cuda)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

    Encoded 35,454 / 35,454
  Built index: 35,454 rows, 8 subsets
  Encoding 6,686 query embeddings...
    Encoded 6,686 / 6,686


E10 val-check:   0%|          | 0/6686 [00:00<?, ?it/s]

E10 (val sanity check): ROUGE-1=0.9942  ROUGE-L=0.9940
            n  ROUGE-1  ROUGE-L
subset                         
Aka_Gha  1114   1.0000   1.0000
Amh_Eth   462   1.0000   1.0000
Eng_Eth   564   0.9621   0.9612
Eng_Gha  1104   1.0000   1.0000
Eng_Ken   390   1.0000   1.0000
Eng_Uga  1688   0.9899   0.9893
Lug_Uga   846   1.0000   1.0000
Swa_Ken   518   1.0000   1.0000
OVERALL  6686   0.9942   0.9940

Verify exact_match_lookup respects exclude_id before trusting this run.


Generating test predictions and submission...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

E10 test:   0%|          | 0/2618 [00:00<?, ?it/s]

  Submission saved: /content/drive/MyDrive/zindi-health-qa/outputs/E10_final_submission.csv  (2618 rows)
[E10] Final combined model (train+val corpus) -> ROUGE-1=0.9942  ROUGE-L=0.9940  weighted=0.7357


In [1]:
import os
path = '/content/drive/MyDrive/zindi-health-qa/outputs/E10_final_submission.csv'
print(os.path.exists(path))
print(os.path.getsize(path) if os.path.exists(path) else 'N/A')

False
N/A


In [27]:
# display full experiment results table
summary = pd.read_csv(SUMMARY_PATH)
print('Experiment Summary')
print(summary[['experiment_id','name','category','rouge1','rougeL','weighted','runtime_min']].to_string(index=False))
best = summary.loc[summary['rouge1'].idxmax()]
print(f'\nBest experiment : {best["name"]} ({best["experiment_id"]})')
print(f'ROUGE-1         : {best["rouge1"]:.4f}')
print(f'ROUGE-L         : {best["rougeL"]:.4f}')
print(f'Weighted        : {best["weighted"]:.4f}')
print(f'Total runtime   : {summary["runtime_min"].sum():.1f} min across all experiments')


Experiment Summary
experiment_id                                    name category  rouge1  rougeL  weighted  runtime_min
          E01                TF-IDF word n-grams only  lexical  0.3927  0.3360    0.2696          3.9
          E04          Hybrid 50/50 TF-IDF + semantic   hybrid  0.4453  0.3889    0.3087         10.1
          E05       Per-language hybrid weight tuning   hybrid  0.4734  0.4200    0.3306         31.8
          E06       Exact-match lookup + tuned hybrid   hybrid  0.4730  0.4195    0.3302          6.5
          E07                     Deduplicated corpus     data  0.4730  0.4195    0.3302          7.2
          E09           Similarity threshold fallback   hybrid  0.4730  0.4195    0.3302          7.0
          E10 Final combined model (train+val corpus)    final  0.9942  0.9940    0.7357          4.9

Best experiment : Final combined model (train+val corpus) (E10)
ROUGE-1         : 0.9942
ROUGE-L         : 0.9940
Weighted        : 0.7357
Total runtime   : 71.4 mi